In [1]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
DA_MAPPINGS_DIR = BASE_DIR / "da_mappings" / "da_mappings_stx"


In [2]:
from IPython.display import display, HTML

rows = []

for fp in sorted(DA_MAPPINGS_DIR.glob("*.json")):
    data = json.loads(fp.read_text(encoding="utf-8"))
    firm_id = fp.stem

    for var_block in data.get("variables", []):
        variable = var_block.get("variable")
        final_choice = var_block.get("final_choice", [])

        if final_choice:  # only keep non-empty final_choice
            for ch in final_choice:
                rows.append({
                    "firm_id": firm_id,
                    "variable": variable,
                    "sheet_name": ch.get("sheet_name", ""),
                    "row_label": ch.get("row_label", ""),
                    "needs_manual_review": var_block.get("needs_manual_review", False),
                    "notes": var_block.get("notes", ""),
                })

df_nonempty = pd.DataFrame(rows)

if df_nonempty.empty:
    print("No non-empty final_choice found.")
else:
    df_show = df_nonempty.sort_values(["variable", "firm_id"]).reset_index(drop=True)

    html_table = df_show.to_html(index=False, escape=False)

    display(HTML(f"""
    <div style="
        max-height: 600px;
        overflow-y: auto;
        overflow-x: auto;
        border: 1px solid #ccc;
        padding: 6px;
        background: black;
    ">
        {html_table}
    </div>
    """))

firm_id,variable,sheet_name,row_label,needs_manual_review,notes
AXFO.ST,COGS_DA,Income Statement,Depreciation & Amortization in Cost of Revenues,False,"Selected only the COGS D&A summary row because the chosen COGS parent row is 'Cost of Goods Sold - Balancing value', which appears net of this separate D&A line. Excluded component rows to avoid double counting."
BONAVb.ST,COGS_DA,Income Statement,Amortization of Intangibles Production costs,False,Selected the separate production-related amortization row because the chosen parent COGS row is 'Production costs' and the amortization appears disclosed separately at the same operating level; using the separate D&A row avoids assuming it is embedded.
DOMETIC.ST,COGS_DA,Income Statement,Depreciation in Cost of Revenues,False,"Selected COGS bucket is only 'Cost of goods sold'. The two cost-of-revenues D&A rows appear as separate operating lines before gross profit, so they should be added separately. No higher-level COGS D&A summary row is visible."
DOMETIC.ST,COGS_DA,Income Statement,Amortization of Intangibles excluding Goodwill in Cost of Revenues,False,"Selected COGS bucket is only 'Cost of goods sold'. The two cost-of-revenues D&A rows appear as separate operating lines before gross profit, so they should be added separately. No higher-level COGS D&A summary row is visible."
DUST.ST,COGS_DA,Income Statement,Depreciation & Amortization in Cost of Revenues,False,Selected the explicit COGS-related D&A row because it is separately disclosed in the operating section and appears outside the chosen COGS row. Did not use any SG&A D&A row here.
ENEA.ST,COGS_DA,Income Statement,Depreciation/Amortization in COR/COGS,False,"Selected only the explicit COGS/COR D&A row because it appears separate from, not embedded in, the chosen COGS bucket."
ESSITYb.ST,COGS_DA,Income Statement,Depreciation in Cost of Revenues,False,"Selected only the separate COGS-related D&A row. It sits outside the chosen 'Cost of goods sold' row and reconciles into total cost of revenues, so adding it separately avoids understating COGS D&A."
HMb.ST,COGS_DA,Income Statement,Depreciation in Cost of Revenues,False,"Selected COGS reference bucket is 'Cost of Operating Revenue'. The statement also shows a separate COGS D&A line before financial items, so it should be added separately. Do not use SG&A depreciation here."
KFASTb.ST,COGS_DA,Income Statement,Depreciation - Investment Property,False,Used only the operating section above financial items. Ignored the broader row 'Depreciation/amortization and impairment' because it mixes impairment with D&A. Selected only the property-specific depreciation line for COGS because it is the clearest separate operating D&A item linked to the rental/investment-property cost bucket and appears outside the previously selected direct-cost rows.
MAHAa.ST,COGS_DA,Income Statement,"Depletion, depreciation and amortization",False,"Selected the explicit populated D&A line because it is presented separately from 'Production costs' in the operating section, implying it should be added separately to COGS rather than treated as already embedded."


In [ ]:
df_manual = df_nonempty[df_nonempty["needs_manual_review"] == True]

print(f"Number of rows:                    {len(df_nonempty)}")
print(f"Unique tickers (firm_id):          {df_nonempty['firm_id'].nunique()}")
print(f"Unique variables:                  {df_nonempty['variable'].nunique()}")
print(f"\nNeeds manual review (total rows):  {len(df_manual)}")
print(f"Needs manual review (unique tickers): {df_manual['firm_id'].nunique()}")
print(f"\nAll tickers: {sorted(df_nonempty['firm_id'].unique().tolist())}")

Rows: 180
Unique tickers (firm_id): 101
